# 3. Evaluación de una red entrenada

Este notebook carga un entrenamiento guardado en `resultados/` y mide qué tan bien maneja su mejor red. No entrena nada.

1. Cómo fue el entrenamiento
2. Cómo es la red: su cromosoma y sus pesos
3. Qué tan bien maneja en un mapa, por tipo de ruta
4. Una ruta paso a paso
5. Generalización: los 58 recortes de la ciudad
6. Comparar entrenamientos
7. Verla manejar en una ventana de pygame

La red siempre maneja con los mismos parámetros con los que se entrenó, como el límite de agarre.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))     # la carpeta del proyecto

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from conduccion import config
from conduccion.aprendizaje.evaluacion import (evaluar_red, manejar, mostrar_resumen, recortes, resumen,
                                               rutas_del_mapa, tabla_por_tipo)
from conduccion.aprendizaje.guardado import cargar, cargar_poblacion, entrenamientos_guardados
from conduccion.aprendizaje.perceptron_multicapa import Perceptron
from conduccion.mapa.mapa import santa_fe
from conduccion.parametros import usar_parametros_de, volver_a_config
from conduccion.visualizacion.graficos import (AZUL, DIVERGENTE, NARANJA, NOMBRES_ENTRADAS, aplicar_estilo,
                                               comparar_entrenamientos, dibujar_mapa, dibujar_recorrido,
                                               dibujar_recorridos_por_tipo, dibujar_ruta, graficar_entrenamiento)

aplicar_estilo()

## 0. Elegir la red y el mapa

`IDENTIFICADOR` es el nombre de la carpeta del entrenamiento en `resultados/`. Por defecto se usa el más reciente. `ZONA` es el mapa donde se prueba, que puede ser distinto del de entrenamiento.

In [ ]:
guardados = entrenamientos_guardados()
if not guardados:
    raise FileNotFoundError("No hay entrenamientos en resultados/. Primero corré 2_entrenamiento.ipynb.")
print("Entrenamientos guardados:", *guardados, sep="\n  ")

IDENTIFICADOR = guardados[-1]
ZONA = "centro"             # "2x2", "3x3", "centro" o "guadalupe"
MAXIMO_RUTAS = 200          # los mapas grandes tienen miles de rutas: se prueban estas

In [ ]:
mejor, historia, datos = cargar(IDENTIFICADOR)
volver_a_config()
distintos = usar_parametros_de(IDENTIFICADOR)

red = Perceptron(config.CAPAS, cromosoma=mejor)
mapa = santa_fe(ZONA)
parametros = datos["parametros"]

print("Entrenamiento:", IDENTIFICADOR)
print("Entrenada en:", parametros["mapa"], "| generaciones:", len(historia), "| población:", parametros["tamano_poblacion"])
print("Cruce:", parametros["tipo_cruce"], "| mutación:", parametros["tipo_mutacion"])
print("Parámetros distintos de config.py:", distintos or "ninguno")
print("Se prueba en:", mapa.nombre)

## 1. Cómo fue el entrenamiento

- **mejor (rutas de la generación):** la nota del mejor individuo en las 3 rutas sorteadas. Sube y baja porque algunas rutas son más fáciles.
- **media de la población.**
- **mejor red (rutas de referencia):** la nota de la mejor red encontrada, en 20 rutas fijas. Es la medida confiable. Casi nunca baja: la mejor red se reemplaza solo si llega en más rutas, y con las mismas llegadas la nota puede bajar un poco.

En un entrenamiento por etapas, este gráfico corresponde a la etapa elegida.

In [ ]:
graficar_entrenamiento(historia, "Entrenamiento en " + parametros["mapa"])

tabla = pd.DataFrame(historia)
ultima_mejora = int(tabla["mejor_referencia"].idxmax())
pd.Series({"Generaciones": len(tabla),
           "Mejor referencia": round(tabla["mejor_referencia"].iloc[-1], 3),
           "Generación de la última mejora": ultima_mejora,
           "Nota media, últimas 20 generaciones": round(tabla["nota_media"].tail(20).mean(), 3),
           "σ": tabla["sigma"].iloc[-1]}, name="valor").to_frame()

## 2. Cómo es la red

El cromosoma, acomodado como la red que lo usa. Cada columna es una neurona y cada fila, lo que llega de una entrada. Azul es un peso positivo y rojo, uno negativo.

In [ ]:
limite = max(abs(red.pesos[0]).max(), abs(red.pesos[1]).max())
fig, ejes = plt.subplots(1, 2, figsize=(11, 5.5), gridspec_kw={"width_ratios": [4, 1.2]})
imagen = ejes[0].imshow(red.pesos[0], cmap=DIVERGENTE, vmin=-limite, vmax=limite, aspect="auto")
ejes[0].set_yticks(range(len(NOMBRES_ENTRADAS)), NOMBRES_ENTRADAS)
ejes[0].set_xlabel("neurona oculta")
ejes[0].set_title("De las entradas a la capa oculta")
ejes[1].imshow(red.pesos[1], cmap=DIVERGENTE, vmin=-limite, vmax=limite, aspect="auto")
ejes[1].set_xticks([0, 1], ["volante", "acelerador"])
ejes[1].set_ylabel("neurona oculta")
ejes[1].set_title("A las salidas")
for ax in ejes:
    ax.grid(False)
fig.colorbar(imagen, ax=ejes, label="valor del peso", shrink=0.8)
plt.show()

¿La población final sigue siendo variada, o ya son casi todos iguales a la mejor red? Si son casi iguales, el cruce ya no aporta y solo queda la mutación.

In [ ]:
poblacion = np.asarray(cargar_poblacion(IDENTIFICADOR)[0])
distancias = np.linalg.norm(poblacion - mejor, axis=1)
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(distancias, bins=20, color=AZUL, edgecolor="white")
ax.set_xlabel("distancia euclídea a la mejor red")
ax.set_ylabel("individuos")
ax.set_title("Diversidad de la población final")
plt.show()

## 3. Qué tan bien maneja en un mapa

La red maneja todas las rutas del mapa de 150 m o más (hasta `MAXIMO_RUTAS`), sin ruido en los sensores. Las rutas se separan según hacia dónde doblan. Como todas las redes se miden en las mismas rutas, esta tabla sirve para compararlas.

In [ ]:
corridas = [manejar(mapa, ruta, red) for ruta in rutas_del_mapa(mapa, MAXIMO_RUTAS)]
pd.DataFrame(tabla_por_tipo(corridas)).set_index("tipo")

In [ ]:
dibujar_recorridos_por_tipo(mapa, corridas)

### Las rutas en las que no llega

Hasta 3 rutas que no llegaron. Para ver otras, se cambia `PRIMERA`.

In [ ]:
PRIMERA = 0
fallas = [corrida for corrida in corridas if corrida["resultado"].motivo != "llegada"]
print("No llega en", len(fallas), "de", len(corridas), "rutas")

elegidas = fallas[PRIMERA:PRIMERA + 3]
if elegidas:
    fig, ejes = plt.subplots(1, len(elegidas), figsize=(5 * len(elegidas), 5), squeeze=False)
    for ax, falla in zip(ejes[0], elegidas):
        resultado = falla["resultado"]
        dibujar_mapa(ax, mapa)
        dibujar_ruta(ax, falla["ruta"], resultado.puntos_alcanzados)
        dibujar_recorrido(ax, resultado, color=NARANJA)
        ax.set_title(falla["tipo"] + ": " + resultado.motivo + " a los " + str(round(resultado.tiempo, 1)) + " s")
    plt.show()

## 4. Una ruta paso a paso

Velocidad, volante, ángulo al próximo punto y calle libre al frente, en cada paso de la ruta. Por defecto se elige la primera ruta con giros a los dos lados.

In [ ]:
elegida = next((c for c in corridas if c["tipo"] == "izquierda y derecha"), corridas[0])
pasos = pd.DataFrame(elegida["pasos"])
print("Tipo:", elegida["tipo"], "| termina por:", elegida["resultado"].motivo,
      "| tiempo:", round(elegida["resultado"].tiempo, 1), "s")

fig = plt.figure(figsize=(14, 7))
ax_mapa = fig.add_subplot(1, 2, 1)
dibujar_mapa(ax_mapa, mapa)
dibujar_ruta(ax_mapa, elegida["ruta"], elegida["resultado"].puntos_alcanzados)
dibujar_recorrido(ax_mapa, elegida["resultado"], color=NARANJA)
ax_mapa.set_title("Recorrido")

series = [("velocidad", "km/h", "Velocidad"), ("volante", "grados", "Volante"),
          ("brujula", "grados", "Ángulo al próximo punto"), ("calle_al_frente", "m", "Calle libre al frente")]
for k, (columna, unidad, titulo) in enumerate(series):
    ax = fig.add_subplot(4, 2, 2 * k + 2)
    ax.plot(pasos["t"], pasos[columna], color=AZUL)
    ax.set_ylabel(unidad)
    ax.set_title(titulo)
ax.set_xlabel("tiempo (s)")
plt.tight_layout()
plt.show()

## 5. Generalización: los 58 recortes de la ciudad

`mapas/recortes.json` tiene 58 recortes de 1 km de toda Santa Fe. Cada uno tiene un nivel de dificultad, de 1 a 4, según la variedad de direcciones de sus calles (entropía de orientación de las calles); los que tienen rotondas van al nivel 4. Uno de cada cinco recortes de cada nivel es de **prueba**: la red nunca entrena ahí. Los niveles sirvieron para elegir los recortes; el informe no los usa.

En cada recorte la red maneja sus 20 rutas de referencia, sin ruido. Cada recorte se evalúa en un proceso aparte. Con todos los núcleos tarda varios minutos.

Lo mismo se puede correr desde la terminal: `python -m conduccion.aprendizaje.evaluacion --red <identificador>`.

In [ ]:
EVALUAR_RECORTES = False        # True para correrlo
NIVELES = None                  # None = todos, o por ejemplo [1, 2]
USO = None                      # None = todos, "entrenamiento" o "prueba"

if EVALUAR_RECORTES:
    por_recorte = evaluar_red(IDENTIFICADOR, recortes(NIVELES, USO))
    mostrar_resumen(por_recorte)
    por_nivel = pd.DataFrame(resumen(por_recorte, "nivel")).T
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(por_nivel.index, 100 * por_nivel["fraccion"], color=AZUL, width=0.6)
    ax.set_xlabel("nivel de dificultad")
    ax.set_ylabel("% de rutas en que llega")
    ax.set_ylim(0, 100)
    ax.set_title("Llegadas por nivel")
    plt.show()

## 6. Comparar entrenamientos

Las últimas corridas guardadas, lado a lado. La nota de referencia solo se puede comparar entre corridas con la misma semilla y el mismo mapa, porque cada una tiene sus propias rutas de referencia. Para una comparación justa se usa la sección 3 o la 5, que miden todas las redes en las mismas rutas.

In [ ]:
ULTIMAS = 4
filas, historias = [], []
for identificador in guardados[-ULTIMAS:]:
    _, h, d = cargar(identificador)
    if not h:
        continue
    historias.append((identificador, h))
    filas.append({"entrenamiento": identificador, "mapa": d["parametros"]["mapa"],
                  "etapa": d["parametros"].get("nombre_etapa", "-"), "semilla": d["parametros"]["semilla"],
                  "cruce": d["parametros"]["tipo_cruce"], "mutación": d["parametros"]["tipo_mutacion"],
                  "agarre": d["config"]["ACELERACION_LATERAL_MAXIMA"], "generaciones": len(h),
                  "mejor referencia": round(h[-1]["mejor_referencia"], 3)})
comparar_entrenamientos(historias)
pd.DataFrame(filas).set_index("entrenamiento")

## 7. Verla manejar

Abre una ventana de pygame donde la red maneja rutas al azar en tiempo real. La ventana puede quedar detrás del editor, y la celda sigue corriendo hasta que se cierra.

Teclas: Espacio = pausa, R = reiniciar la ruta, N = ruta nueva, S = rayos, flechas = más o menos rápido, Esc = salir. En pausa, un clic en una esquina y otro en otra eligen la ruta.

In [ ]:
from conduccion.visualizacion.simulador_pygame import ver_simulacion

ver_simulacion(mapa, red)